# LLM setup



In [1]:
!pip uninstall -y datasets


In [2]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IN_COLAB = "google.colab" in sys.modules


def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None

# Skipped once the packages are installed, so this cell can be re-run after restarting the session.
if IN_COLAB and installed_version("pandasai") != "3.0.0":
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Choose a Python 3.11 Colab runtime before installing these packages.")
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    # Same versions as uv.lock (torch is preinstalled on Colab).
    packages = [
        "ipython>=8,<9",
        "matplotlib-inline<0.2",
        "numpy==1.26.4",
        "pandas==2.3.3",
        "transformers==5.17.0",
        "accelerate==1.15.0",
        "bitsandbytes==0.50.2",
        "peft==0.21.2",
        "sentence-transformers==6.1.0",
        "requests",
        "pandasai==3.0.0",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Choose Runtime → Restart session before continuing.")

In [8]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 101.3 MB/s eta 0:00:00


## Get the project code from github

On Colab, clone your team's repository so that the `ift6758` package can be imported.
For a private repository, create a GitHub [personal access token](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens) with read access and add it to Colab's *Secrets* (key icon in the left sidebar) as `GITHUB_TOKEN`. Never paste the token in the notebook.

In [3]:
import os
import subprocess
IN_COLAB=True
if IN_COLAB:
    from google.colab import userdata

    REPO = "aliciavaq/data_science_project"
    REPO_DIR = "/content/repo"
    if not os.path.exists(REPO_DIR):
        token = userdata.get("GITHUB_TOKEN")
        subprocess.check_call(["git", "clone", f"https://{token}@github.com/{REPO}.git", REPO_DIR])
    sys.path.insert(0, REPO_DIR)

## Check the GPU

In [4]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU found. On Colab, select a T4 GPU runtime (Runtime → Change runtime type).")

GPU: Tesla T4


## Load the model

The first run downloads ~8 GB of weights to the Hugging Face cache.

In [5]:
from ift6758.llm import load_model

model, tokenizer = load_model()
print(f"Loaded on {model.device} ({model.dtype})")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded on cuda:0 (torch.float16)


In [6]:
messages = [{"role": "user", "content": "In one sentence, what is a hat trick in hockey?"}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)
output_ids = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In hockey, a hat trick occurs when a player scores a goal, records an assist, and gets a penalty in the same game.  

*Note: While the term "hat trick" is commonly associated with scoring three goals, in hockey it is more accurately defined as scoring a goal, an assist, and a penalty


#Code Generation
Using the retrieved documentation, prompt the provided Qwen 4B model to write a Python class or function that fetches, caches locally, and loads play-by-play data for a given season.

##RAG from NHL docs implementation


In [13]:
from ift6758.llm.rag import LLM_DIR, embedder,load_docs, build_index, retrieve, save_retrieval

docs = load_docs()

chunks, index = build_index(docs)
print(f"{len(chunks)} chunks indexed")

query ="fetch play-by-play information for a specific game"
context = retrieve(query, chunks, embedder, index, k=4)
save_retrieval(LLM_DIR / "retrieved.md", query, context)
print(context)



100 chunks indexed
[1] ### Game Events
#### Get Play By Play
- **Endpoint**: `/v1/gamecenter/{game-id}/play-by-play`
- **Method**: GET
- **Description**: Retrieve play-by-play information for a specific game.
- **Parameters**:
  - `game-id` (int) - Game ID
- **Response**: JSON format

###### Example using cURL:

```bash
curl -X GET "https://api-web.nhle.com/v1/gamecenter/2023020204/play-by-play"
```

#### Get Landing
- **Endpoint**: `/v1/gamecenter/{game-id}/landing`
- **Method**: GET
- **Description**: Retrieve landing information for a specific game.
- **Parameters**:
  - `game-id` (int) - Game ID
- **Response**: JSON format

###### Example using cURL:

```bash
curl -X GET "https://api-web.nhle.com/v1/gamecenter/2023020204/landing"
```

[2] ### Additional Game Content

#### Get Game Right Rail Content
- **Endpoint**: `/v1/gamecenter/{game-id}/right-rail`
- **Method**: GET
- **Description**: Retrieves sidebar content for the game center view.
- **Parameters**:
  - `game-id` (int) - Ga